# 모의고사 03. 와인 등급 분류 (다중분류)

> 실제 시험처럼 이론 설명 없이 문제만 순서대로 제시합니다. **90분 타이머를 맞추고** 풀어보세요. 오픈북 허용 문서(numpy/pandas/matplotlib/seaborn/scikit-learn/tensorflow/XGBoost 공식문서)만 참고할 수 있다고 가정합니다.

### 데이터 소개
`data/wine.csv` - 와인의 화학 성분(alcohol, malic_acid 등)으로 품종(target 0/1/2)을 분류하는 다중분류 문제입니다.

> ⏱️ **[실전 타임어택 가이드]** 데이터 탐색 10분 ➔ 전처리 20분 ➔ 머신러닝 30분 ➔ 딥러닝 30분
> 막히는 부분은 주석으로 남기고 다음 문제로 넘어가는 것이 합격 전략입니다.


## 목차
1교시 데이터 로딩 & EDA · 2교시 데이터 시각화 · 3교시 데이터 전처리 · 4교시 머신러닝 모델링 · 5교시 딥러닝 모델링 · 채점 가이드

In [ ]:
import sys
sys.path.insert(0, '../00_시작하기')
import aice_grader as grader

# 실전처럼 시간 제한(90분)을 두고 풀어보세요. 아래 셀을 실행하면 타이머가 시작됩니다.
timer = grader.ExamTimer(minutes=90)
timer.start()

## 1교시: 데이터 로딩 & EDA

**문제 1.** `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`을 각각 `pd`, `np`, `plt`, `sns`로 불러오고, `../data/wine.csv`을 `wine`로 불러와 shape을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

wine = pd.read_csv('../data/wine.csv')
wine.head()
print(wine.shape)
```

</details>

**문제 2.** `wine`의 shape과 `target` 값 분포를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(wine.shape)
print(wine['target'].value_counts())  # 클래스별 데이터 개수를 확인해 불균형 여부를 미리 체크
```

</details>

**문제 3.** 수치형 변수 중 `target`과 상관관계가 가장 강한 변수를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
corr = wine.corr()['target'].drop('target')  # target 자기 자신과의 상관계수(항상 1)는 제외
print(corr.abs().sort_values(ascending=False).head())  # abs()로 절댓값을 취해야 음의 상관관계도 강한 관계로 함께 비교됨
```

</details>

## 2교시: 데이터 시각화

**문제 4.** `target`별 개수를 countplot으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.countplot(x='target', data=wine)  # 클래스별 빈도를 막대로 확인
plt.show()
```

</details>

**문제 5.** `alcohol`을 `target`별로 boxplot으로 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.boxplot(x='target', y='alcohol', data=wine)  # 품종(target)별로 alcohol 분포가 어떻게 다른지 비교
plt.show()
```

</details>

## 3교시: 데이터 전처리

**문제 6.** `target`을 y, 나머지를 X로 나누고 `train_test_split(test_size=0.2, stratify=y, random_state=42)`로 분할하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import train_test_split
X = wine.drop(columns=['target'])
y = wine['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # stratify=y로 클래스 비율을 train/test에 동일하게 유지
print(X_train.shape)
```

</details>

**문제 7.** `StandardScaler`로 스케일링하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # train에만 fit
X_test_s = scaler.transform(X_test)  # test는 transform만(데이터 누수 방지)
```

</details>

## 4교시: 머신러닝 모델링

**문제 8.** `RandomForestClassifier(n_estimators=100, random_state=42)`를 학습시키고 `classification_report`를 출력하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_s, y_train)
print(classification_report(y_test, rf.predict(X_test_s)))  # average 옵션 없이도 클래스별 precision/recall/f1을 모두 보여줘 다중분류에서 가장 먼저 찍어보기 좋음
```

</details>

**문제 9.** f1-score를 `average='macro'`와 `'weighted'`로 각각 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import f1_score
pred = rf.predict(X_test_s)
print(f1_score(y_test, pred, average='macro'))  # macro: 클래스 크기 무시하고 단순 평균
print(f1_score(y_test, pred, average='weighted'))  # weighted: 클래스별 데이터 개수로 가중 평균
```

</details>

**문제 10.** 3x3 confusion matrix를 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, pred)  # 클래스가 3개이므로 3x3 행렬이 반환됨
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
plt.show()
```

</details>

## 5교시: 딥러닝 모델링

**문제 11.** 출력층 노드=3, activation='softmax'인 Sequential 모델을 만들고 `sparse_categorical_crossentropy`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(X_train_s.shape[1],)))
model.add(Dense(8, activation='relu'))
model.add(Dense(3, activation='softmax'))  # 다중분류 출력층: 노드 수=클래스 수(3), softmax
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # y가 정수 레이블이므로 sparse_categorical_crossentropy 사용
```

</details>

**문제 12.** 학습 후 `np.argmax`로 최종 클래스를 구해 accuracy를 계산하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train_s, y_train, epochs=100, batch_size=16, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
pred_label = np.argmax(model.predict(X_test_s), axis=1)  # softmax 확률 벡터를 argmax로 최종 클래스 라벨로 변환
print(accuracy_score(y_test, pred_label))
```

</details>

In [ ]:
# 문제를 다 풀었다면 아래 셀을 실행해 소요 시간을 확인하세요.
timer.finish()

## 채점 가이드 (100점 만점 배점표)

이 모의고사는 총 12문제, 100점 만점입니다. 각 문제를 정답과 비교해 맞았으면 배점만큼, 틀렸으면 0점으로 스스로 채점해보세요.

| 문항 | 배점 |
|---|---|
| 문제 1 | 8점 |
| 문제 2 | 8점 |
| 문제 3 | 8점 |
| 문제 4 | 8점 |
| 문제 5 | 8점 |
| 문제 6 | 8점 |
| 문제 7 | 8점 |
| 문제 8 | 8점 |
| 문제 9 | 9점 |
| 문제 10 | 9점 |
| 문제 11 | 9점 |
| 문제 12 | 9점 |
| **합계** | **100점** |

> 💡 **합격 기준: 80점 이상** (실제 AICE Associate와 동일한 기준입니다)

### 자동으로 기록하며 채점하고 싶다면

`00_시작하기/aice_grader.py`의 `MockExamGrader`를 사용하면 점수를 자동으로 합산해줍니다.

```python
import aice_grader as grader

g = grader.MockExamGrader("모의고사03_와인등급분류")
g.check(1, points=8, is_correct=True)   # 문제 1을 맞았으면 True, 틀렸으면 False
g.check(2, points=8, is_correct=True)
# ... 문제 수만큼 반복 ...
g.report()   # 최종 점수와 합격 여부를 출력
```

### 개념 체크리스트

다 풀었다면 아래 체크리스트로 한 번 더 점검해보세요.

- [ ] 라이브러리를 정확한 alias(pd, np, plt, sns 등)로 불러왔는가
- [ ] 다중분류 평가지표에 average 옵션을 지정했는가
- [ ] 출력층 노드 수가 클래스 수(3)와 같은가
- [ ] softmax + sparse_categorical_crossentropy 조합을 썼는가
- [ ] argmax로 예측 후처리를 했는가
